In [ ]:
!pip install lightkurve torch numpy

In [ ]:
%%writefile data_pipeline.py
import lightkurve as lk
import numpy as np
import logging

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

def get_clean_lightcurve(tic_id: str) -> tuple[np.ndarray, np.ndarray]:
    logger.info(f"Searching for TESS light curve for {tic_id}...")
    search_result = lk.search_lightcurve(tic_id, mission='TESS', author='SPOC')
    
    if len(search_result) == 0:
        return np.array([])
        
    # VERY IMPORTANT: Only download the first sector!
    # Downloading all sectors uses too much RAM and crashes Google Colab.
    lc = search_result[0].download()
    if lc is None:
        return np.array([])
        
    clean_lc = lc.remove_nans().remove_outliers().flatten(window_length=101)
    
    time_1d = clean_lc.time.value
    flux_1d = clean_lc.flux.value
    
    return time_1d, flux_1d


In [ ]:
%%writefile model_training.py
import torch
import torch.nn as nn

INPUT_LENGTH = 20000

class ExoplanetCNN(nn.Module):
    def __init__(self, input_length=INPUT_LENGTH):
        super(ExoplanetCNN, self).__init__()
        self.input_length = input_length
        self.conv_layers = nn.Sequential(
            nn.Conv1d(in_channels=1, out_channels=16, kernel_size=7, stride=2, padding=3),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2),
            nn.Conv1d(in_channels=16, out_channels=32, kernel_size=7, stride=2, padding=3),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2),
            nn.Conv1d(in_channels=32, out_channels=64, kernel_size=7, stride=1, padding=3),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2),
        )
        self._to_linear = None
        self._get_conv_output_size()
        self.fc_layers = nn.Sequential(
            nn.Linear(self._to_linear, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 3) 
        )

    def _get_conv_output_size(self):
        with torch.no_grad():
            dummy_input = torch.zeros(1, 1, self.input_length)
            output = self.conv_layers(dummy_input)
            self._to_linear = output.numel()

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layers(x)
        return x


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from data_pipeline import get_clean_lightcurve
from model_training import ExoplanetCNN, INPUT_LENGTH

def process_flux(flux_array, target_length):
    if len(flux_array) == 0:
        return None
    
    # Convert astropy MaskedNDArray to standard numpy array to prevent np.pad crashes
    flux_array = np.array(flux_array, dtype=np.float32)
    
    if len(flux_array) > target_length:
        return flux_array[:target_length]
    elif len(flux_array) < target_length:
        pad_size = target_length - len(flux_array)
        pad_value = np.median(flux_array) if len(flux_array) > 0 else 1.0
        return np.pad(flux_array, (0, pad_size), 'constant', constant_values=pad_value)
    
    return flux_array

def main():
    planet_tics = [
        "TIC 279741379", "TIC 281541555", "TIC 149090620", 
        "TIC 280053912", "TIC 162239401"
    ]
    
    noise_tics = [
        "TIC 149258286", "TIC 149179973", "TIC 149089947", 
        "TIC 280054000", "TIC 149090630"
    ]
    
    X, y = [], []
    
    print("Downloading and processing planet TICs (Label 0)...")
    for tic in planet_tics:
        result = get_clean_lightcurve(tic)
        if isinstance(result, tuple) and len(result) == 2:
            time_arr, flux_arr = result
            flux_processed = process_flux(flux_arr, INPUT_LENGTH)
            if flux_processed is not None:
                X.append(flux_processed)
                y.append(0)  
        else:
            print(f"Failed to process {tic}.")

    print("Downloading and processing noise/binary TICs (Labels 1 and 2)...")
    for i, tic in enumerate(noise_tics):
        result = get_clean_lightcurve(tic)
        if isinstance(result, tuple) and len(result) == 2:
            time_arr, flux_arr = result
            flux_processed = process_flux(flux_arr, INPUT_LENGTH)
            if flux_processed is not None:
                X.append(flux_processed)
                label = 1 if i < 2 else 2
                y.append(label)
        else:
            print(f"Failed to process {tic}.")

    if len(X) == 0:
        print("No valid data collected. Exiting.")
        return

    X = np.array(X)
    y = np.array(y)
    
    X_tensor = torch.tensor(X, dtype=torch.float32).unsqueeze(1)
    y_tensor = torch.tensor(y, dtype=torch.long)
    
    dataset = TensorDataset(X_tensor, y_tensor)
    dataloader = DataLoader(dataset, batch_size=4, shuffle=True)
    
    model = ExoplanetCNN(input_length=INPUT_LENGTH)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    epochs = 20
    print(f"\nStarting training for {epochs} epochs...")
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for inputs, labels in dataloader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * inputs.size(0)
            
        epoch_loss = running_loss / len(dataset)
        print(f"Epoch {epoch+1}/{epochs} - Loss: {epoch_loss:.4f}")
        
    save_path = 'exoplanet_model.pth'
    torch.save(model.state_dict(), save_path)
    print(f"\nTraining complete. Model weights successfully saved to '{save_path}'.")

if __name__ == "__main__":
    main()
